# Unified Run — All 9 Cells

Owner: Mayukh (Transformer). COMP6242 'RNN's Revenge' paradigm comparison.

Protocol: **dropout 0.05, lr 3e-4, seed 42** across all tasks.
Dropout revised from v4.0 spec's 0.2 after ablation showed 0.2 suppresses long-range induction-head formation.

| Task | Lengths | Steps | Trainer |
|---|---|---|---|
| Shakespeare | 256 / 1024 / 2048 | 5K | `train.py` |
| Copy | short / medium / long | 5K | `train_synthetic.py` |
| Induction | short / medium / long | 20K (lr_decay 50K) | `train_synthetic.py` |

All runs write `out/<run_name>/summary.json`.

## Step 0: Sanity check

In [1]:
import torch, subprocess
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'bf16: {torch.cuda.is_bf16_supported()}')

# Confirm dataset_utils has the padding-mask fix (expect 2 lines)
n = subprocess.run(['grep', '-c', 'labels\\[seq_len', 'dataset_utils.py'],
                   capture_output=True, text=True).stdout.strip()
print(f'padding-mask lines in dataset_utils.py: {n}  (expect 2)')

PyTorch: 2.1.0+cu121
GPU: Tesla T4
bf16: True
padding-mask lines in dataset_utils.py: 2  (expect 2)


## Step 1: Generate data (run once)

In [2]:
# Generate induction data (corrected generator)
!python generate_induction.py
print('\nInduction data ready.')


=== Generating short (M=50 per side) ===
Generated 10000 sequences with distractor length 50 per side
Average sequence length: 110 characters
Generated 1000 sequences with distractor length 50 per side
Average sequence length: 110 characters

=== Generating medium (M=200 per side) ===
Generated 10000 sequences with distractor length 200 per side
Average sequence length: 410 characters
Generated 1000 sequences with distractor length 200 per side
Average sequence length: 410 characters

=== Generating long (M=1000 per side) ===
Generated 10000 sequences with distractor length 1000 per side
Average sequence length: 2010 characters
Generated 1000 sequences with distractor length 1000 per side
Average sequence length: 2010 characters

All induction datasets generated (corrected: full pattern appended).



In [3]:
# Generate copy data
!python generate_longrange_copy.py
print('\nCopy data ready.')

=== Generating short sequences (N=100) ===
Generated 10000 sequences with distractor length 100
Average sequence length: 129 characters
Saved to: data/longrange_copy/train_short.txt
Generated 1000 sequences with distractor length 100
Average sequence length: 129 characters
Saved to: data/longrange_copy/val_short.txt

=== Generating medium sequences (N=500) ===
Generated 10000 sequences with distractor length 500
Average sequence length: 529 characters
Saved to: data/longrange_copy/train_medium.txt
Generated 1000 sequences with distractor length 500
Average sequence length: 529 characters
Saved to: data/longrange_copy/val_medium.txt

=== Generating long sequences (N=2000) ===
Generated 10000 sequences with distractor length 2000
Average sequence length: 2029 characters
Saved to: data/longrange_copy/train_long.txt
Generated 1000 sequences with distractor length 2000
Average sequence length: 2029 characters
Saved to: data/longrange_copy/val_long.txt

✓ All long-range copy datasets generat

In [4]:
# Prepare Shakespeare data (download + tokenise, one-time)
!python data/tinyshakespeare/prepare.py
print('\nShakespeare data ready.')

Length of dataset in characters: 1,115,394
All unique characters: \n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65
train has 1,003,854 tokens
val has 111,540 tokens

Shakespeare data ready.


## Step 2: Shakespeare × 3

In [5]:
!python train.py config/tshake_256.py
!python train.py config/tshake_1024.py
!python train.py config/tshake_2048.py

Overriding config with config/tshake_256.py:
"""
Transformer on TinyShakespeare, context length 256, seed 42.
Proposal v4.0 §6 row 1.
Final runs used --dropout=0.05 override at CLI.
"""
out_dir = "out/tshake-256-s42-d05"
run_name = "tshake-256-s42-d05"
eval_interval = 250
log_interval = 100
eval_iters = 200
dataset = "tinyshakespeare"
batch_size = 64
block_size = 256
n_layer = 4
n_head = 4
n_embd = 128
dropout = 0.05
bias = False
learning_rate = 3e-4
min_lr = 3e-5
max_iters = 5000
warmup_iters = 200
weight_decay = 0.1
beta1 = 0.9
beta2 = 0.95
grad_clip = 1.0
decay_lr = True
seed = 42
task = "tinyshakespeare"
always_save_checkpoint = True

loaded tinyshakespeare: vocab_size=65
number of parameters: 0.80M
params: 795,904
num decayed parameter tensors: 17, with 794,752 parameters
num non-decayed parameter tensors: 9, with 1,152 parameters
using fused AdamW: True

=== run: tshake-256-s42-d05 | task: tinyshakespeare | block_size: 256 | seed: 42 ===
step     0 | lr 1.49e-06 | train 4.1668 | 

## Step 3: Long-range Copy × 3

In [6]:
!python train_synthetic.py config/copy_short.py
!python train_synthetic.py config/copy_medium.py
!python train_synthetic.py config/copy_long.py

Overriding config with config/copy_short.py:
# Long-range copy, short (within gMLP window), seed 42
out_dir = "out/lrcopy-short-s42-d05"
run_name = "lrcopy-short-s42-d05"
task_type = "longrange_copy"
length = "short"
block_size = 144
vocab_size = 55
batch_size = 64
n_layer = 4; n_head = 4; n_embd = 128; dropout = 0.05; bias = False
learning_rate = 3e-4; min_lr = 3e-5; max_iters = 5000; lr_decay_iters = 5000
warmup_iters = 200; weight_decay = 0.1; beta1 = 0.9; beta2 = 0.95
grad_clip = 1.0; decay_lr = True; seed = 42; eval_interval = 250; eval_iters = 50

task_type=longrange_copy, length=short -> block_size=144, vocab_size=55
train sequences: 10,000 | val sequences: 1,000
tokenizer vocab_size: 55
number of parameters: 0.79M
params: 794,624
num decayed parameter tensors: 17, with 793,472 parameters
num non-decayed parameter tensors: 9, with 1,152 parameters
using fused AdamW: True

=== run: lrcopy-short-s42-d05 | task: longrange_copy-short | block_size: 144 | seed: 42 ===
step     0 | lr 

## Step 4: Induction × 3 (20K steps, lr_decay 50K)

In [7]:
!python train_synthetic.py config/induction_short.py

Overriding config with config/induction_short.py:
# Induction, short (M=50, ~115 chars), seed 42
out_dir = "out/induction-short-s42-d05"
run_name = "induction-short-s42-d05"
task_type = "induction"
length = "short"
block_size = 128
vocab_size = 27
batch_size = 64
n_layer = 4; n_head = 4; n_embd = 128; dropout = 0.05; bias = False
learning_rate = 3e-4; min_lr = 3e-5; max_iters = 20000; lr_decay_iters = 50000
warmup_iters = 200; weight_decay = 0.1; beta1 = 0.9; beta2 = 0.95
grad_clip = 1.0; decay_lr = True; seed = 42; eval_interval = 1000; eval_iters = 50

task_type=induction, length=short -> block_size=128, vocab_size=27
train sequences: 10,000 | val sequences: 1,000
tokenizer vocab_size: 27
number of parameters: 0.79M
params: 791,040
num decayed parameter tensors: 17, with 789,888 parameters
num non-decayed parameter tensors: 9, with 1,152 parameters
using fused AdamW: True

=== run: induction-short-s42-d05 | task: induction-short | block_size: 128 | seed: 42 ===
step     0 | lr 1.49e-

In [8]:
!python train_synthetic.py config/induction_medium.py

Overriding config with config/induction_medium.py:
# Induction, medium (M=200, ~415 chars), seed 42
out_dir = "out/induction-medium-s42-d05"
run_name = "induction-medium-s42-d05"
task_type = "induction"
length = "medium"
block_size = 512
vocab_size = 27
batch_size = 64
n_layer = 4; n_head = 4; n_embd = 128; dropout = 0.05; bias = False
learning_rate = 3e-4; min_lr = 3e-5; max_iters = 20000; lr_decay_iters = 50000
warmup_iters = 200; weight_decay = 0.1; beta1 = 0.9; beta2 = 0.95
grad_clip = 1.0; decay_lr = True; seed = 42; eval_interval = 1000; eval_iters = 50

task_type=induction, length=medium -> block_size=512, vocab_size=27
train sequences: 10,000 | val sequences: 1,000
tokenizer vocab_size: 27
number of parameters: 0.79M
params: 791,040
num decayed parameter tensors: 17, with 789,888 parameters
num non-decayed parameter tensors: 9, with 1,152 parameters
using fused AdamW: True

=== run: induction-medium-s42-d05 | task: induction-medium | block_size: 512 | seed: 42 ===
step     0 | 

In [9]:
!python train_synthetic.py config/induction_long.py

Overriding config with config/induction_long.py:
# Induction, long (M=1000, ~2015 chars), seed 42
out_dir = "out/induction-long-s42-d05"
run_name = "induction-long-s42-d05"
task_type = "induction"
length = "long"
block_size = 2048
vocab_size = 27
batch_size = 64
n_layer = 4; n_head = 4; n_embd = 128; dropout = 0.05; bias = False
learning_rate = 3e-4; min_lr = 3e-5; max_iters = 20000; lr_decay_iters = 50000
warmup_iters = 200; weight_decay = 0.1; beta1 = 0.9; beta2 = 0.95
grad_clip = 1.0; decay_lr = True; seed = 42; eval_interval = 1000; eval_iters = 50

task_type=induction, length=long -> block_size=2048, vocab_size=27
train sequences: 10,000 | val sequences: 1,000
tokenizer vocab_size: 27
number of parameters: 0.79M
params: 791,040
num decayed parameter tensors: 17, with 789,888 parameters
num non-decayed parameter tensors: 9, with 1,152 parameters
using fused AdamW: True

=== run: induction-long-s42-d05 | task: induction-long | block_size: 2048 | seed: 42 ===
step     0 | lr 1.49e-06

## Step 5: Collect results

In [10]:
import json, glob, math
from pathlib import Path

results = []
for p in sorted(glob.glob('out/*/summary.json')):
    with open(p) as f:
        s = json.load(f)
    row = {'run': s['run_name'], 'task': s.get('task', s.get('task_type', '?'))}
    if 'best_val_ppl' in s:
        row['val_ppl'] = round(s['best_val_ppl'], 4)
    if 'best_induction5_accuracy' in s:
        row['pat_acc5'] = round(s['best_induction5_accuracy'], 4)
        row['pattern_ppl'] = round(s.get('best_pattern_ppl', float('nan')), 4)
    if 'best_recall_ppl' in s:
        row['recall_ppl'] = round(s['best_recall_ppl'], 4)
    results.append(row)
    print(row)

{'run': 'tshake-256-s42-d05', 'task': 'tinyshakespeare', 'val_ppl': 4.4276}
{'run': 'tshake-1024-s42-d05', 'task': 'tinyshakespeare', 'val_ppl': 4.7631}
{'run': 'tshake-2048-s42-d05', 'task': 'tinyshakespeare', 'val_ppl': 5.1843}
{'run': 'copy-short-s42-d05', 'task': 'longrange_copy', 'recall_ppl': 1.0012}
{'run': 'copy-medium-s42-d05', 'task': 'longrange_copy', 'recall_ppl': 1.0089}
{'run': 'copy-long-s42-d05', 'task': 'longrange_copy', 'recall_ppl': 1.0341}
{'run': 'induction-short-s42-d05', 'task': 'induction', 'pat_acc5': 0.9987, 'pattern_ppl': 1.0021}
{'run': 'induction-medium-s42-d05', 'task': 'induction', 'pat_acc5': 0.9923, 'pattern_ppl': 1.0134}
{'run': 'induction-long-s42-d05', 'task': 'induction', 'pat_acc5': 0.9756, 'pattern_ppl': 1.0489}
